In [1]:
import torch
import torch.nn as nn

class GRUSequenceClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        
        # 1. Embedding Layer to map token integers to continuous vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. Optimized GRU Engine
        # batch_first=True expects inputs with shape: (Batch, Sequence Length, Features)
        self.gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        
        # 3. Dense Classification Output Head
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        # x shape: (Batch, Seq_Len)
        embedded = self.embedding(x)  # Shape becomes: (Batch, Seq_Len, Embedding_Dim)
        
        # Pass tokens through the GRU engine
        # out: hidden states for every time step across the sequence
        # hn: the absolute final terminal hidden state tensor (No separate cell state tensor exists!)
        out, hn = self.gru(embedded)
        
        # Extract the final step's hidden state to capture full structural sequence summary
        # hn shape: (num_layers, batch, hidden_dim) -> extract layer 0
        final_hidden = hn[-1]
        
        # Map to class logits
        logits = self.fc(final_hidden)
        return logits

# Instantiate the model (e.g., 5000 vocab size, 128 embedding, 256 hidden units, 2 classes)
model = GRUSequenceClassifier(vocab_size=5000, embedding_dim=128, hidden_dim=256, num_classes=2)
print(model)

# Audit with a mock text batch (4 samples, each containing exactly 20 sequence indices)
mock_batch = torch.randint(low=0, high=5000, size=(4, 20))
output_logits = model(mock_batch)

print(f"\nForward GRU tracking successful!")
print(f"Output Matrix Shape: {output_logits.shape} (Batch Size, Target Classes)")

GRUSequenceClassifier(
  (embedding): Embedding(5000, 128)
  (gru): GRU(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=2, bias=True)
)

Forward GRU tracking successful!
Output Matrix Shape: torch.Size([4, 2]) (Batch Size, Target Classes)
